# 🧬 Notebook 02 — Advanced Synthetic Data Generation (Literature-Driven)

## Obiettivo
Generare **100.000 pazienti sintetici** con serie temporali di 14 giorni di check-in
emotivi sulla scala SINTON-IA (10 stati d'animo × 10 livelli di intensità).

## Filosofia: "Nucleo Clinico + Rumore Empirico"
La logica di generazione si regge su due pilastri con priorità precise:

1. **NUCLEO (Letteratura Scientifica)**: Tutti i parametri fondamentali (medie di valence,
   tassi di dropout, prevalenza depressiva, persistenza emotiva) sono definiti come
   **costanti immutabili** derivate dalla letteratura psicologica peer-reviewed.

2. **RUMORE (Dataset Reali)**: I pattern estratti nel Notebook 01 dai dataset
   StudentLife e 14-Day AA contribuiscono **esclusivamente** la varianza statistica
   (deviazioni standard) e le matrici di transizione di Markov, per conferire
   naturalezza e realismo alle serie temporali generate.

### Perché questo approccio?
I dataset reali presentano limiti significativi che ne precludono l'uso diretto:
- **StudentLife**: Campione ristretto di studenti universitari americani (~48 soggetti),
  contesto altamente specifico (stress accademico), scarsa generalizzabilità.
- **14-Day AA**: Struttura invertita rispetto al nostro task (PHQ-9 somministrato
  quotidianamente + happiness score — non umore giornaliero → PHQ-9 finale).

## Riferimenti Scientifici
Ogni parametro clinico è tracciabile a una fonte peer-reviewed specifica.

## 1. Import e Caricamento Pattern Empirici

In [1]:
import os
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
np.random.seed(42)

RAW_DIR = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
FIGURES_DIR = os.path.join('..', '..', '..', 'docs', 'documentazione', 'latex', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

# Caricamento pattern empirici dal Notebook 01 (ruolo: SOLO varianza e transizioni)
with open(os.path.join(RAW_DIR, 'statistical_patterns.json'), 'r', encoding='utf-8') as f:
    patterns = json.load(f)

VA_MAP = patterns['sintonia_valence_arousal_map']
EMPIRICAL_STATS = patterns['stats_by_severity']
TRANS_MATRICES = patterns['transition_matrices']
TRANS_CATEGORIES = patterns['transition_categories']

print("✅ Pattern empirici caricati (ruolo: varianza e transizioni Markov)")
print(f"   • {len(VA_MAP)} stati d'animo SINTON-IA mappati")
print(f"   • {len(EMPIRICAL_STATS)} fasce di severità con statistiche")
print(f"   • {len(TRANS_MATRICES)} matrici di transizione estratte dai dati reali")

✅ Pattern empirici caricati (ruolo: varianza e transizioni Markov)
   • 10 stati d'animo SINTON-IA mappati
   • 5 fasce di severità con statistiche
   • 5 matrici di transizione estratte dai dati reali


---
## 2. Parametri Clinici dalla Letteratura Scientifica

### 2.1 Costanti Fondamentali
Ogni parametro è una **costante immutabile** derivata da studi peer-reviewed.
I dataset reali non influenzano questi valori di base.

In [2]:
# ═══════════════════════════════════════════════════════════════
# PARAMETRI CLINICI (LETTERATURA SCIENTIFICA — NUCLEO IMMUTABILE)
# ═══════════════════════════════════════════════════════════════

N_PATIENTS = 100_000
N_DAYS = 14
MOOD_STATES = list(VA_MAP.keys())
N_INTENSITY_LEVELS = 10

# ── 2.1 DISTRIBUZIONE PHQ-9 NELLA POPOLAZIONE ──────────────────
# Fonte: WHO (2023) — Prevalenza globale ~3.8%; studi italiani mostrano
#        distribuzione simile (De Girolamo et al., 2012).
# Nota: La distribuzione è intenzionalmente sbilanciata verso le fasce
#       basse, riflettendo la realtà epidemiologica.
PHQ9_DISTRIBUTION = {
    'Minima (0-4)':        0.55,   # 55%  — WHO (2023)
    'Lieve (5-9)':         0.25,   # 25%  — WHO (2023)
    'Moderata (10-14)':    0.12,   # 12%  — De Girolamo et al. (2012)
    'Mod. Severa (15-19)': 0.05,   # 5%   — De Girolamo et al. (2012)
    'Severa (20-27)':      0.03,   # 3%   — De Girolamo et al. (2012)
}

# ── 2.2 VALENCE MEDIA PER FASCIA (COSTANTI DA LETTERATURA) ─────
# Fonte primaria: Kuppens et al. (2010) — L'inerzia emotiva è
#   significativamente più alta nei soggetti con sintomi depressivi.
# Fonte secondaria: Houben et al. (2015) — Meta-analisi che conferma
#   la correlazione tra dinamiche emotive negative e malessere.
# I valori di base riflettono il tono affettivo medio atteso per
# ciascuna fascia di gravità, nello spazio Valence [-1, +1].
LITERATURE_VALENCE_PARAMS = {
    'Minima (0-4)':        {'mean':  0.45, 'std': 0.25},
    'Lieve (5-9)':         {'mean':  0.15, 'std': 0.30},
    'Moderata (10-14)':    {'mean': -0.10, 'std': 0.30},
    'Mod. Severa (15-19)': {'mean': -0.35, 'std': 0.25},
    'Severa (20-27)':      {'mean': -0.60, 'std': 0.20},
}

# ── 2.3 DROPOUT RATE (MNAR) PER FASCIA ─────────────────────────
# Fonte: Karyotaki et al. (2015) — I tassi di dropout nelle
#   app di salute mentale crescono con la severità depressiva.
# Fonte: Mohr et al. (2017) — Il personal sensing conferma che
#   l'aderenza al monitoraggio cala nei pazienti più gravi.
DROPOUT_RATES = {
    'Minima (0-4)':        0.05,   # 5%  — Karyotaki et al. (2015)
    'Lieve (5-9)':         0.10,   # 10% — Karyotaki et al. (2015)
    'Moderata (10-14)':    0.18,   # 18% — Mohr et al. (2017)
    'Mod. Severa (15-19)': 0.28,   # 28% — Mohr et al. (2017)
    'Severa (20-27)':      0.40,   # 40% — Karyotaki et al. (2015)
}

# ── 2.4 SMILING DEPRESSION ─────────────────────────────────────
# Fonte: Helmich et al. (2020) — Alcuni pazienti con depressione
#   severa mostrano pattern non lineari con apparenti miglioramenti
#   temporanei che mascherano la gravità reale.
# La "smiling depression" è il fenomeno per cui il paziente riporta
# stati d'animo positivi nonostante un PHQ-9 elevato.
SMILING_DEPRESSION_RATE = {
    'Minima (0-4)':        0.00,
    'Lieve (5-9)':         0.02,
    'Moderata (10-14)':    0.07,
    'Mod. Severa (15-19)': 0.10,   # Helmich et al. (2020)
    'Severa (20-27)':      0.08,   # Helmich et al. (2020)
}

# ── 2.5 APPIATTIMENTO AFFETTIVO ────────────────────────────────
# Fonte: Rottenberg et al. (2005) — I pazienti depressi mostrano
#   "emotion context insensitivity", ossia le loro reazioni emotive
#   risultano attenuate sia in contesti positivi che negativi.
AFFECTIVE_BLUNTING = {
    'Minima (0-4)':        1.00,
    'Lieve (5-9)':         0.98,
    'Moderata (10-14)':    0.92,
    'Mod. Severa (15-19)': 0.85,   # Rottenberg et al. (2005)
    'Severa (20-27)':      0.80,   # Rottenberg et al. (2005)
}

# ── 2.6 PERSISTENZA DEGLI STATI NEGATIVI ───────────────────────
# Fonte: aan het Rot et al. (2012) — Le streak negative consecutive
#   durano mediamente 3-7 giorni nei pazienti con depressione.
# Fonte: Wichers et al. (2016) — Il "critical slowing down" indica
#   che i sistemi vicini alla transizione patologica mostrano
#   maggiore inerzia (recupero lento dagli stati negativi).

print("═" * 60)
print("PARAMETRI CLINICI (DEFINITI DA LETTERATURA SCIENTIFICA)")
print("═" * 60)
print(f"\n  Pazienti da generare:  {N_PATIENTS:>10,}")
print(f"  Giorni per paziente:   {N_DAYS:>10}")
print(f"  Stati d'animo:         {len(MOOD_STATES):>10}")
print(f"  Livelli intensità:     {N_INTENSITY_LEVELS:>10}")
print(f"\nDistribuzione PHQ-9 (WHO, 2023; De Girolamo et al., 2012):")
for sev, pct in PHQ9_DISTRIBUTION.items():
    bar = '█' * int(pct * 40)
    print(f"  {sev:<22} {pct*100:5.1f}%  {bar}")


════════════════════════════════════════════════════════════
PARAMETRI CLINICI (DEFINITI DA LETTERATURA SCIENTIFICA)
════════════════════════════════════════════════════════════

  Pazienti da generare:     100,000
  Giorni per paziente:           14
  Stati d'animo:                 10
  Livelli intensità:             10

Distribuzione PHQ-9 (WHO, 2023; De Girolamo et al., 2012):
  Minima (0-4)            55.0%  ██████████████████████
  Lieve (5-9)             25.0%  ██████████
  Moderata (10-14)        12.0%  ████
  Mod. Severa (15-19)      5.0%  ██
  Severa (20-27)           3.0%  █


---
## 3. Funzioni di Supporto: Integrazione Letteratura + Dati Empirici

In [3]:
# ═══════════════════════════════════════════════════════════════
# 3.1  GENERAZIONE PHQ-9 REALISTICO
# ═══════════════════════════════════════════════════════════════

def generate_phq9_score(severity):
    """
    Genera un punteggio PHQ-9 realistico per una data fascia di severità.
    Utilizza distribuzioni Beta(2,2) troncate per evitare artefatti ai bordi.
    Fonte soglie: Kroenke et al. (2001); Manea et al. (2012).
    """
    ranges = {
        'Minima (0-4)':        (0, 4),
        'Lieve (5-9)':         (5, 9),
        'Moderata (10-14)':    (10, 14),
        'Mod. Severa (15-19)': (15, 19),
        'Severa (20-27)':      (20, 27),
    }
    lo, hi = ranges[severity]
    raw = np.random.beta(2, 2)
    return int(lo + raw * (hi - lo + 1))


def valence_to_mood_state(valence, arousal):
    """
    Dato un punto (Valence, Arousal), trova lo stato SINTON-IA più vicino
    nello spazio circomplesso usando la distanza euclidea.
    Fonte modello: Russell (1980); Posner et al. (2005).
    """
    min_dist = float('inf')
    best_state = 'Neutro'

    for state, coords in VA_MAP.items():
        v, a = coords['valence'], coords['arousal']
        dist = np.sqrt((valence - v)**2 + (arousal - a)**2)
        if dist < min_dist:
            min_dist = dist
            best_state = state

    return best_state


def get_valence_params(severity):
    """
    Restituisce i parametri (media, std) della Valence per una fascia.

    LOGICA DI PRIORITÀ:
    - MEDIA: sempre dalla LETTERATURA (Kuppens et al., 2010; Houben et al., 2015)
    - STD: dalla letteratura come base, MODULATA dalla std empirica dei
           dataset reali (StudentLife / 14-Day) se disponibile.
    """
    lit = LITERATURE_VALENCE_PARAMS[severity]
    base_mean = lit['mean']
    base_std = lit['std']

    # La std empirica aggiunge varianza "umana" (se disponibile)
    if severity in EMPIRICAL_STATS:
        empirical_std = EMPIRICAL_STATS[severity].get('valence_std', base_std)
        # Blend: 70% letteratura, 30% empirico
        final_std = 0.7 * base_std + 0.3 * empirical_std
    else:
        final_std = base_std

    return base_mean, final_std


def generate_intensity(valence, severity):
    """
    Genera il livello di intensità (1-10) basandosi sulla valenza
    e sulla severità del paziente.

    Il fenomeno dell'appiattimento affettivo (Rottenberg et al., 2005)
    riduce l'intensità nei pazienti depressi: le emozioni vengono
    vissute in modo meno intenso sia in contesti positivi che negativi.
    """
    base_intensity = abs(valence) * 6 + 3  # Range ~3-9
    damping = AFFECTIVE_BLUNTING.get(severity, 1.0)
    base_intensity *= damping
    intensity = base_intensity + np.random.normal(0, 2.0)
    return int(np.clip(round(intensity), 1, 10))


def get_transition_matrix(severity):
    """
    Restituisce la matrice di transizione di Markov per la fascia.

    Le matrici provengono dai dataset reali (Notebook 01) e catturano
    l'inerzia emotiva naturale (Kuppens et al., 2010): la tendenza
    a rimanere nello stesso stato emotivo da un giorno all'altro.

    Se non disponibili per una fascia, ne viene generata una sintetica
    con elevata persistenza diagonale (~40%), come suggerito dalla
    letteratura sull'emotional inertia.
    """
    if severity in TRANS_MATRICES:
        return np.array(TRANS_MATRICES[severity])

    # Fallback: matrice sintetica con inerzia elevata
    n = len(TRANS_CATEGORIES)
    mat = np.eye(n) * 0.4
    for i in range(n):
        for j in range(n):
            if i != j:
                distance = abs(i - j)
                mat[i, j] = 0.6 / (distance * n)
    mat = mat / mat.sum(axis=1, keepdims=True)
    return mat


---
## 4. Motore di Generazione Principale

In [4]:
# ═══════════════════════════════════════════════════════════════
# 4.  GENERAZIONE PAZIENTE (ARCHITETTURA A 3 LIVELLI)
# ═══════════════════════════════════════════════════════════════

def generate_patient(patient_id, severity, is_smiling=False):
    """
    Genera una serie temporale completa di 14 giorni per un singolo paziente.

    Architettura a 3 livelli:
    - Livello 1: Profilo paziente (PHQ-9, parametri individuali)
    - Livello 2: Catena di Markov emotiva (transizioni giorno-giorno)
    - Livello 3: Rumore clinico (MNAR dropout + smiling depression)

    Ogni livello è governato da parametri della letteratura scientifica,
    con varianza addizionale dai dataset empirici.
    """
    phq9 = generate_phq9_score(severity)
    v_mean, v_std = get_valence_params(severity)

    # Smiling depression: shift positivo della valence riportata
    # Fonte: Helmich et al. (2020)
    if is_smiling:
        v_mean = min(v_mean + 0.4, 0.3)
        v_std *= 0.7

    # Matrice di transizione (dai dati reali — ruolo: naturalezza)
    trans_mat = get_transition_matrix(severity)

    # Dropout rate individuale (dalla letteratura + varianza personale)
    # Fonte base: Karyotaki et al. (2015); Mohr et al. (2017)
    base_dropout = DROPOUT_RATES.get(severity, 0.10)
    patient_dropout = np.clip(base_dropout + np.random.normal(0, 0.03), 0.01, 0.60)

    # Stato iniziale
    current_valence = np.clip(np.random.normal(v_mean, v_std), -1, 1)
    current_category_idx = min(int((current_valence + 1) / 2 * len(TRANS_CATEGORIES)),
                               len(TRANS_CATEGORIES) - 1)

    records = []
    for day in range(N_DAYS):
        # --- STEP 1: Dropout MNAR (Karyotaki et al., 2015) ---
        # Il dropout aumenta con le streak negative consecutive
        # Fonte: aan het Rot et al. (2012) — la persistenza negli
        #   stati negativi correla con il disengagement
        consecutive_negative = 0
        for prev in reversed(records):
            if prev.get('mood_state') is not None and prev.get('valence', 0) < -0.2:
                consecutive_negative += 1
            else:
                break

        adjusted_dropout = patient_dropout * (1 + consecutive_negative * 0.15)
        adjusted_dropout = min(adjusted_dropout, 0.70)

        if np.random.random() < adjusted_dropout:
            records.append({
                'patient_id': patient_id, 'day': day + 1,
                'mood_state': None, 'intensity': None,
                'valence': None, 'arousal': None, 'is_missing': True
            })
            continue

        # --- STEP 2: Transizione Markoviana (dai dati reali) ---
        # Fonte inerzia: Kuppens et al. (2010)
        probs = trans_mat[current_category_idx]
        probs = probs / probs.sum()
        new_category_idx = np.random.choice(len(TRANS_CATEGORIES), p=probs)
        current_category_idx = new_category_idx

        # --- STEP 3: Valence continua dalla categoria ---
        # Media dalla LETTERATURA, perturbazione dal processo stocastico
        cat_centers = {0: -0.8, 1: -0.4, 2: 0.0, 3: 0.4, 4: 0.8}
        cat_valence = cat_centers[new_category_idx]
        perturbation = np.random.normal(0, 0.15)
        current_valence = np.clip(cat_valence + perturbation + (v_mean * 0.3), -1, 1)

        # --- STEP 4: Arousal ---
        arousal_base = abs(current_valence) * 0.5
        if current_valence < 0:
            arousal_base *= 0.7
        arousal_noise = np.random.normal(0, 0.15)
        current_arousal = np.clip(arousal_base + arousal_noise, -1, 1)

        # --- STEP 5: Mappa V-A → Stato SINTON-IA (Russell, 1980) ---
        mood_state = valence_to_mood_state(current_valence, current_arousal)
        intensity = generate_intensity(current_valence, severity)

        records.append({
            'patient_id': patient_id, 'day': day + 1,
            'mood_state': mood_state, 'intensity': intensity,
            'valence': round(current_valence, 4),
            'arousal': round(current_arousal, 4), 'is_missing': False
        })

    for r in records:
        r['phq9_total'] = phq9
        r['severity'] = severity
        r['is_smiling_depression'] = is_smiling

    return records

---
## 5. Esecuzione della Generazione

In [5]:
print("🧬 Inizio generazione sintetica...")
print(f"   Target: {N_PATIENTS:,} pazienti × {N_DAYS} giorni = {N_PATIENTS*N_DAYS:,} record\n")

all_records = []
severity_counts = {sev: 0 for sev in PHQ9_DISTRIBUTION}
smiling_count = 0
start_time = datetime.now()
generated_patients = 0

while generated_patients < N_PATIENTS:
    severity = np.random.choice(
        list(PHQ9_DISTRIBUTION.keys()),
        p=list(PHQ9_DISTRIBUTION.values())
    )
    is_smiling = np.random.random() < SMILING_DEPRESSION_RATE.get(severity, 0)

    patient_records = generate_patient(generated_patients, severity, is_smiling)
    
    # Garantiamo che ogni paziente abbia almeno 3 check-in (per il filtro nel NB 03)
    n_present = sum(1 for r in patient_records if not r['is_missing'])
    if n_present >= 3:
        all_records.extend(patient_records)
        severity_counts[severity] += 1
        if is_smiling:
            smiling_count += 1
        generated_patients += 1

        if generated_patients % 20000 == 0:
            elapsed = (datetime.now() - start_time).total_seconds()
            rate = generated_patients / elapsed
            remaining = (N_PATIENTS - generated_patients) / rate
            print(f"  ▶ {generated_patients:>7,} / {N_PATIENTS:,} pazienti generati "
                  f"({(generated_patients)/N_PATIENTS*100:.0f}%) — "
                  f"~{remaining:.0f}s rimanenti")

elapsed_total = (datetime.now() - start_time).total_seconds()
print(f"\n✅ Generazione completata in {elapsed_total:.1f} secondi")
print(f"   Record totali generati: {len(all_records):,}")
print(f"   Casi smiling depression: {smiling_count:,} ({smiling_count/N_PATIENTS*100:.2f}%)")

🧬 Inizio generazione sintetica...
   Target: 100,000 pazienti × 14 giorni = 1,400,000 record



  ▶  20,000 / 100,000 pazienti generati (20%) — ~269s rimanenti


  ▶  40,000 / 100,000 pazienti generati (40%) — ~201s rimanenti


  ▶  60,000 / 100,000 pazienti generati (60%) — ~140s rimanenti


  ▶  80,000 / 100,000 pazienti generati (80%) — ~69s rimanenti


  ▶ 100,000 / 100,000 pazienti generati (100%) — ~0s rimanenti

✅ Generazione completata in 329.7 secondi
   Record totali generati: 1,400,000
   Casi smiling depression: 2,124 (2.12%)


In [6]:
df_synthetic = pd.DataFrame(all_records)
print(f"\nDataFrame sintetico: {df_synthetic.shape[0]:,} righe × {df_synthetic.shape[1]} colonne")
print(f"\nColonne: {list(df_synthetic.columns)}")
df_synthetic.head(10)


DataFrame sintetico: 1,400,000 righe × 10 colonne

Colonne: ['patient_id', 'day', 'mood_state', 'intensity', 'valence', 'arousal', 'is_missing', 'phq9_total', 'severity', 'is_smiling_depression']


,patient_id,day,mood_state,intensity,valence,arousal,is_missing,phq9_total,severity,is_smiling_depression
0,0,1,NaN,NaN,NaN,NaN,True,1,Minima (0-4),False
1,0,2,Felice,7.0,0.8479,0.3452,False,1,Minima (0-4),False
2,0,3,Neutro,2.0,0.3964,0.3207,False,1,Minima (0-4),False
3,0,4,Neutro,2.0,0.0295,-0.3062,False,1,Minima (0-4),False
4,0,5,Neutro,3.0,0.2247,0.1307,False,1,Minima (0-4),False
5,0,6,Confuso,6.0,-0.3550,0.2664,False,1,Minima (0-4),False
6,0,7,Confuso,5.0,-0.3603,0.0461,False,1,Minima (0-4),False
7,0,8,Confuso,10.0,-0.4548,0.3230,False,1,Minima (0-4),False
8,0,9,Felice,9.0,0.7140,0.3898,False,1,Minima (0-4),False
9,0,10,Neutro,5.0,-0.0164,-0.2318,False,1,Minima (0-4),False


---
## 6. Validazione del Dataset Generato

In [7]:
# ═══════════════════════════════════════════════════════════════
# 6.1  DISTRIBUZIONE SEVERITÀ GENERATA vs TARGET
# ═══════════════════════════════════════════════════════════════
patients_df = df_synthetic.drop_duplicates(subset='patient_id')

print("═" * 65)
print("CONFRONTO DISTRIBUZIONE SEVERITÀ: TARGET vs GENERATA")
print("═" * 65)
print(f"{'Fascia':<22} {'Target':>8} {'Generata':>10} {'Delta':>8}")
print("─" * 65)
for sev in PHQ9_DISTRIBUTION:
    target = PHQ9_DISTRIBUTION[sev] * 100
    actual = (patients_df['severity'] == sev).mean() * 100
    delta = actual - target
    print(f"{sev:<22} {target:>7.1f}% {actual:>9.1f}% {delta:>+7.2f}%")

═════════════════════════════════════════════════════════════════
CONFRONTO DISTRIBUZIONE SEVERITÀ: TARGET vs GENERATA
═════════════════════════════════════════════════════════════════
Fascia                   Target   Generata    Delta
─────────────────────────────────────────────────────────────────
Minima (0-4)              55.0%      55.2%   +0.15%
Lieve (5-9)               25.0%      24.9%   -0.12%
Moderata (10-14)          12.0%      12.0%   +0.02%
Mod. Severa (15-19)        5.0%       4.9%   -0.05%
Severa (20-27)             3.0%       3.0%   +0.00%


In [8]:
# ═══════════════════════════════════════════════════════════════
# 6.2  DISTRIBUZIONE STATI D'ANIMO PER FASCIA
# ═══════════════════════════════════════════════════════════════
non_missing = df_synthetic[~df_synthetic['is_missing']]

fig, axes = plt.subplots(1, 5, figsize=(25, 5), sharey=True)
severities = list(PHQ9_DISTRIBUTION.keys())

for idx, sev in enumerate(severities):
    subset = non_missing[non_missing['severity'] == sev]
    if len(subset) > 0:
        mood_counts = subset['mood_state'].value_counts()
        mood_pcts = (mood_counts / len(subset) * 100).reindex(MOOD_STATES, fill_value=0)

        colors = ['#FFD700','#87CEEB','#FF6347','#C0C0C0','#8B8B83',
                  '#4169E1','#FF8C00','#DC143C','#9400D3','#DDA0DD']

        axes[idx].barh(range(len(MOOD_STATES)), mood_pcts.values, color=colors)
        axes[idx].set_yticks(range(len(MOOD_STATES)))
        axes[idx].set_yticklabels(MOOD_STATES if idx == 0 else [])
        axes[idx].set_xlabel('%')
        axes[idx].set_title(sev.split('(')[0].strip(), fontsize=10, fontweight='bold')

plt.suptitle('Distribuzione Stati d\'Animo per Fascia di Severità', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'dp_mood_distribution.png'), dpi=150, bbox_inches='tight')
plt.close()
print("✅ Figura salvata: dp_mood_distribution.png")

✅ Figura salvata: dp_mood_distribution.png


In [9]:
# ═══════════════════════════════════════════════════════════════
# 6.3  TASSO DI MISSING DATA (MNAR) PER FASCIA
# ═══════════════════════════════════════════════════════════════
print("\n═" * 60)
print("TASSO DI MISSING DATA (MNAR) — GENERATO vs ATTESO")
print("═" * 60)
print(f"{'Fascia':<22} {'Atteso':>8} {'Generato':>10}")
print("─" * 50)
for sev in severities:
    subset = df_synthetic[df_synthetic['severity'] == sev]
    actual_missing = subset['is_missing'].mean() * 100
    expected = DROPOUT_RATES.get(sev, 0) * 100
    print(f"{sev:<22} {expected:>7.1f}% {actual_missing:>9.1f}%")


═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
TASSO DI MISSING DATA (MNAR) — GENERATO vs ATTESO
════════════════════════════════════════════════════════════
Fascia                   Atteso   Generato
──────────────────────────────────────────────────
Minima (0-4)               5.0%       5.3%


Lieve (5-9)               10.0%      10.2%
Moderata (10-14)          18.0%      19.1%
Mod. Severa (15-19)       28.0%      30.4%
Severa (20-27)            40.0%      43.2%


In [10]:
# ═══════════════════════════════════════════════════════════════
# 6.4  CORRELAZIONE VALENCE-PHQ9 (Verifica Anti-Leakage)
# ═══════════════════════════════════════════════════════════════
patient_stats = non_missing.groupby('patient_id').agg(
    valence_mean=('valence', 'mean'),
    valence_std=('valence', 'std'),
    phq9=('phq9_total', 'first'),
    is_smiling=('is_smiling_depression', 'first')
).reset_index()

corr = patient_stats['valence_mean'].corr(patient_stats['phq9'])
print(f"\n═" * 60)
print(f"VERIFICA DATA LEAKAGE")
print(f"═" * 60)
print(f"  Correlazione Valence_media ↔ PHQ-9: {corr:.4f}")
if abs(corr) > 0.95:
    print(f"  ⚠️ ATTENZIONE: Correlazione troppo alta — rischio data leakage!")
elif abs(corr) > 0.80:
    print(f"  ✅ Correlazione forte ma non perfetta — dataset realistico")
else:
    print(f"  ✅ Correlazione moderata — buona variabilità, no leakage")

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(patient_stats['phq9'], patient_stats['valence_mean'],
                     c=patient_stats['is_smiling'].map({True: 'red', False: '#3498db'}),
                     alpha=0.05, s=5)
ax.set_xlabel('Punteggio PHQ-9', fontsize=12)
ax.set_ylabel('Valence Media (14 giorni)', fontsize=12)
ax.set_title(f'PHQ-9 vs Valence Media — r = {corr:.3f}', fontsize=14, fontweight='bold')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

smiling_pts = patient_stats[patient_stats['is_smiling']]
if len(smiling_pts) > 0:
    ax.scatter(smiling_pts['phq9'], smiling_pts['valence_mean'],
              c='red', alpha=0.3, s=10, label=f'Smiling Depression (n={len(smiling_pts)})')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'dp_leakage_check.png'), dpi=150, bbox_inches='tight')
plt.close()
print("✅ Figura salvata: dp_leakage_check.png")


═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
═
VERIFICA DATA LEAKAGE
════════════════════════════════════════════════════════════
  Correlazione Valence_media ↔ PHQ-9: -0.6165
  ✅ Correlazione moderata — buona variabilità, no leakage


✅ Figura salvata: dp_leakage_check.png


---
## 7. Salvataggio del Dataset Sintetico

In [11]:
output_path = os.path.join(RAW_DIR, 'synthetic_depression_data_v2.csv')
df_synthetic.to_csv(output_path, index=False, encoding='utf-8')

file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"✅ Dataset sintetico salvato in: {output_path}")
print(f"   Dimensione: {file_size_mb:.1f} MB")
print(f"   Record totali: {len(df_synthetic):,}")
print(f"   Pazienti unici: {df_synthetic['patient_id'].nunique():,}")
print(f"   Missing data: {df_synthetic['is_missing'].sum():,} ({df_synthetic['is_missing'].mean()*100:.1f}%)")

gen_metadata = {
    'generated_at': datetime.now().isoformat(),
    'approach': 'Literature-Driven with Empirical Variance',
    'literature_sources': {
        'valence_params': 'Kuppens et al. (2010); Houben et al. (2015)',
        'dropout_rates': 'Karyotaki et al. (2015); Mohr et al. (2017)',
        'smiling_depression': 'Helmich et al. (2020)',
        'affective_blunting': 'Rottenberg et al. (2005)',
        'emotional_inertia': 'Kuppens et al. (2010); Wichers et al. (2016)',
        'negative_streaks': 'aan het Rot et al. (2012)',
        'phq9_scoring': 'Kroenke et al. (2001); Manea et al. (2012)',
        'circumplex_model': 'Russell (1980); Posner et al. (2005)',
        'prevalence': 'WHO (2023); De Girolamo et al. (2012)',
    },
    'empirical_sources': 'StudentLife (Wang et al., 2014) + 14-Day AA (Mueller, 2023)',
    'empirical_role': 'Transition matrices and variance modulation only',
    'n_patients': int(N_PATIENTS),
    'n_days': int(N_DAYS),
    'total_records': int(len(df_synthetic)),
    'missing_records': int(df_synthetic['is_missing'].sum()),
    'smiling_patients': int(smiling_count),
    'valence_phq9_correlation': float(corr),
}

meta_path = os.path.join(PROCESSED_DIR, 'generation_metadata.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(gen_metadata, f, indent=2, ensure_ascii=False)

print(f"\n✅ Metadati generazione salvati in: {meta_path}")

✅ Dataset sintetico salvato in: ..\data\raw\synthetic_depression_data_v2.csv
   Dimensione: 80.4 MB
   Record totali: 1,400,000
   Pazienti unici: 100,000
   Missing data: 147,777 (10.6%)

✅ Metadati generazione salvati in: ..\data\processed\generation_metadata.json
